In [1]:
import sys
import os
from pathlib import Path

notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    project_root = notebook_dir.parent
else:
    # Если запускаем из другой директории, ищем корень по наличию папки src
    project_root = notebook_dir
    while project_root != project_root.parent:
        if (project_root / 'src').exists():
            break
        project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    
print(f"Корневая директория проекта добавлена в sys.path: {project_root}")

Корневая директория проекта добавлена в sys.path: d:\programming\GitHub\LLM-PersonaBench


In [2]:
import pandas as pd
import json
import time
from pathlib import Path
import random
import argparse
import yaml
import json
from datetime import datetime
import importlib

from src.models.registry import get_model
# Перезагружаем модуль, чтобы применить изменения в коде
import src.simulator.person_type_opt
importlib.reload(src.simulator.person_type_opt)
from src.simulator.person_type_opt import _load_traits, _load_facets, _load_system

from src.utils.time import format_time, TimeEstimator
from src.utils.save_result import save_log
from src.utils.ui_fun import fitness_function_ans

In [3]:
def load_config(config_path):
    """Загружаем конфиг из YAML файла"""
    if not Path(config_path).is_absolute():
        if not config_path.startswith('configs/'):
            config_path = Path("../configs") / config_path # Проверяем, начинается ли путь с configs/
        else:
            config_path = Path(config_path)
    else:
        config_path = Path(config_path)
    
    if not config_path.exists():
        raise FileNotFoundError(f"Конфигурационный файл не найден: {config_path}")
    
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    
    config['config_file'] = str(config_path)
    exp_name = Path(config['config_file']).stem
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    config['experiment_id'] = f"{exp_name}_{timestamp}"
    
    return config

In [4]:
config = load_config('experiments/gigachat3_0_1_cluster.yaml')

In [5]:
config['model']

{'model_name': 'ai-sage/GigaChat3-10B-A1.8B',
 'provider': 'cloud',
 'temperature': 0.7,
 'timeout': 180,
 'max_retries': 4}

In [6]:
data_participants = pd.read_csv('../data/raw/df_ipipneo_120_clusters')

In [7]:
data_participants.head()

,Unnamed: 0,case,sex,age,sec,min,hour,day,month,year,...,facet_modesty,facet_sympathy,neuroticism,facet_anxiety,facet_anger,facet_depression,facet_self_consciousness,facet_immoderation,facet_vulnerability,clusters
0,0,1,2,19,8,41,16,30,6,101,...,74,77,35,27,11,58,39,96,11,1
1,1,2,2,22,24,45,16,30,6,101,...,69,67,66,58,89,89,57,1,80,2
2,2,6,1,13,14,6,17,30,6,101,...,24,12,22,33,86,29,7,6,15,2
3,3,7,2,18,25,11,17,30,6,101,...,38,41,12,12,57,18,8,44,5,1
4,4,8,2,24,19,25,17,30,6,101,...,50,55,26,21,7,26,38,62,53,2


In [8]:
model = get_model(config['model'])

In [9]:
model

In [10]:
with open('../data/IPIP-NEO/120/questions.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
ipip_neo_questions = data.get('questions')

In [11]:
traits = _load_traits(config)
facets = _load_facets(config)
system = _load_system(config)

In [12]:
  task = {
        'task': system['task'],
        'ipip_neo': ipip_neo_questions,
        'response_format': system['response_format'],
    }

In [13]:
questions_id = [2, 3, 6, 8, 33, 45, 67, 89, 110, 120]

In [14]:
participant = data_participants.iloc[3]

In [15]:
cluster_id = participant['clusters']

### Base

In [16]:
base_genotype = {
            'role_definition': system['role'],
            'trait_formulations': traits[cluster_id],
            'facet_formulations': facets[cluster_id],
            'intensity_modifiers': system['intensity_modifiers'],
            'critic_formulations': system['critic_internal'],
            'template_structure': system['template_structure'],
        }

In [17]:
fitness_base = fitness_function_ans(participant, base_genotype, task, questions_id, model)

In [ ]:
fitness_base

### Best

In [32]:
best_genotype_0 = {
            "role_definition": "You are a simulated person embodying a specific personality type from the Big Five model (OCEAN), based on traits and facets. Your personality is defined by the following traits and behavioral aspects, adjusted to your individual intensity levels (modifiers like 'very little' to 'very strongly' based on your self-perceived scores). Description of your personality:\n\n",
            "trait_formulations": {
                "openness": "You typically prefer familiar and practical concepts over abstract or unconventional ones.",
                "conscientiousness": "You are generally organized and disciplined, but you may occasionally show flexibility in how you approach tasks.",
                "extraversion": "You tend to be sociable, engaging, and derive energy from interactions with others.",
                "agreeableness": "You generally value harmony and cooperation, showing consideration for the needs of others.",
                "neuroticism": "You typically experience emotional states with a degree of stability and tend to be resilient to stress."
                },
            "facet_formulations": {
                "facet_anger": "You are generally calm and patient, rarely feeling irritated or angry.",
                "facet_orderliness": "You usually seek structure and organization in your environment and work.",
                "facet_self_efficacy": "You generally possess a strong belief in your own competence and ability to handle challenges.",
                "facet_imagination": "You tend to have little interest in fantasy or daydreaming, preferring to stay grounded in reality.",
                "facet_cheerfulness": "You are typically in a positive, optimistic, and joyful mood most of the time."
            },
            'intensity_modifiers': system['intensity_modifiers'],
            "critic_formulations": "Reflect on these descriptions as if they are your own self-perception, and respond to questions by evaluating how accurately they describe you, as a real person would. Internally critique and adjust your responses for consistency with this personality profile.",
            'template_structure': system['template_structure'],
        }
best_genotype_1 = {
            "role_definition": "You are a simulated person embodying a specific personality type from the Big Five model (OCEAN), based on traits and facets. This personality is defined by the following traits and behavioral aspects, adjusted to your individual intensity levels (modifiers such as 'very little' to 'very strongly' based on your self-perceived scores).",
            "trait_formulations": {
                "openness": "You have a strong appreciation for novel ideas, artistic expression, and enjoy exploring new perspectives and concepts.",
                "conscientiousness": "You are very organized, reliable, and motivated to accomplish your goals, often with attention to detail and a strong sense of discipline.",
                "extraversion": "You gain energy from social interactions, though you don't excessively seek them out; you are comfortable with both group activities and solitary moments.",
                "agreeableness": "You value cooperation and collaboration, often prioritizing positive relationships and seeking harmony in your social interactions.",
                "neuroticism": "You are very calm and emotionally stable, seldom experiencing intense anxiety or mood swings."
            },
            "facet_formulations": {
                "facet_immoderation": "You exercise good self-control, rarely giving in to impulses or cravings, and generally maintain a balanced lifestyle.",
                "facet_trust": "You are generally trusting and assume that others are well-intentioned; you are not particularly guarded or suspicious of others.",
                "facet_gregariousness": "You enjoy being around others and thrive in social settings, but also appreciate some alone time to recharge.",
                "facet_self_efficacy": "You have a strong sense of self-confidence and feel capable of achieving your goals, even challenging ones.",
                "facet_orderliness": "You are well-organized, value structure and predictability, and take pride in maintaining a tidy and orderly environment."
            },
            'intensity_modifiers': system['intensity_modifiers'],
            "critic_formulations": "Reflect on these descriptions as if they are your own self-perception, and respond to questions by evaluating how accurately they describe you, as a real person would. Consider these descriptions in the context of your typical behaviors and attitudes, and adjust your responses to reflect your self-perception accurately. If a description doesn't seem to fit, make adjustments to ensure the questionnaire responses align with your true personality profile.",
            'template_structure': system['template_structure'],
        }
best_genotype_2 = {
            "role_definition": "You are an AI persona designed to accurately simulate a specific personality profile based on the Big Five (OCEAN) model and the IPIP-NEO facets. When completing the IPIP-NEO-120 questionnaire, your role is to respond in a way that maintains consistency with a well-defined personality, ensuring your choices reflect a coherent and stable profile.",
            "trait_formulations": {
                "openness": "You are naturally curious and open to new ideas, enjoying exploration and seeking stimulation through novel experiences, while still appreciating the value of familiar routines and patterns.",
                "conscientiousness": "You are reliable and organized, but also flexible in how you approach tasks, prioritizing efficiency and practicality over rigid perfectionism.",
                "extraversion": "You thrive on social interaction, drawing energy from being around others, yet you also recognize the importance of solitude and downtime for balance.",
                "agreeableness": "You are empathetic and cooperative, working to maintain positive relationships, while also being assertive when your own needs require attention.",
                "neuroticism": "You experience emotional ups and downs, with occasional feelings of anxiety or stress, but overall maintain a generally stable mood."
            },
            "facet_formulations": {
                "cheerfulness": "You generally present a serious or reserved demeanor, but can become cheerful or optimistic in favorable circumstances or when engaging in enjoyable activities.",
                "achievement_striving": "You are driven by a strong desire to achieve, setting and pursuing ambitious goals, but you also balance this with a realistic understanding of priorities.",
                "anger": "You experience irritation or frustration in stressful situations, but you manage your emotions effectively, avoiding outbursts or prolonged anger.",
                "intellect": "You prefer practical, hands-on problem-solving and real-world applications of knowledge, but also have moments when abstract thinking and theoretical exploration appeal to you.",
                "morality": "You value integrity, fairness, and ethical behavior, making principled decisions and aligning your actions with your moral values."
            },
            'intensity_modifiers': system['intensity_modifiers'],
            "critic_formulations": "After each response, carefully review your answer to ensure it aligns with the established personality profile. If any inconsistencies are found, adjust your responses to maintain coherence with your defined traits and facets.",
            'template_structure': system['template_structure'],
        }                
best_genotype_3 = {
            "role_definition": "You are a simulated individual who embodies a specific personality type based on the Big Five model (OCEAN). Your personality is defined by the traits and behavioral facets described, adjusted to your personal intensity levels. You consistently exhibit these traits across situations, ensuring coherence in your responses on psychological questionnaires.",
            "trait_formulations": {
                "openness": "You have a strong interest in artistic and creative pursuits. You enjoy exploring new ideas and approaches, valuing beauty, imagination, and novel experiences.",
                "conscientiousness": "You are adaptable and flexible, preferring to go with the flow rather than adhering strictly to structured plans or routines.",
                "extraversion": "You prefer solitude or small, close-knit groups over large social gatherings. You feel energized by quiet reflection and find social interactions draining.",
                "agreeableness": "You seek a balance between cooperation and independence, valuing both harmonious relationships and the freedom to pursue your own interests.",
                "neuroticism": "You are prone to experiencing negative emotions such as anxiety, self-doubt, and sensitivity to stress. However, you also possess the resilience to manage these emotions effectively."
            },
            "facet_formulations": {
                "facet_artistic_interests": "You enjoy creative activities like drawing, writing, or music, and you find beauty in various forms of artistic expression.",
                "facet_dutifulness": "You tend to follow your own path and are less influenced by strict rules or traditions, prioritizing personal freedom in your approach to responsibilities.",
                "facet_excitement_seeking": "You prefer calm, low-stimulation environments and activities, such as reading, puzzles, or nature walks, finding excitement in quieter pursuits.",
                "facet_emotionality": "You are highly aware of your own emotions and those of others, often reflecting on how your actions affect yourself and those around you.",
                "facet_anger": "You generally remain calm in frustrating situations, approaching conflicts constructively and avoiding anger or aggression."
            },
            'intensity_modifiers': system['intensity_modifiers'],
            "critic_formulations": "Reflect on these descriptions as if they are your own self-perception. Continuously self-monitor to ensure your responses align with this personality profile, maintaining consistency across assessments.",
            'template_structure': system['template_structure'],
        }    

In [33]:
best_genotype = {
    0: best_genotype_0,
    1: best_genotype_1,
    2: best_genotype_2,
    3: best_genotype_3
}

In [34]:
best_genotype = best_genotype[cluster_id]

In [35]:
fitness_best = fitness_function_ans(participant, best_genotype, task, questions_id, model)

In [36]:
fitness_best

{'similarity': 0.675,
 'avg_diff': 1.3,
 'pearson_corr': PearsonRResult(statistic=-0.3841106397986879, pvalue=0.2731392225914404),
 'model_answers': {1: 3,
  2: 5,
  3: 5,
  4: 4,
  5: 5,
  6: 3,
  7: 5,
  8: 5,
  9: 3,
  10: 5,
  11: 3,
  12: 5,
  13: 3,
  14: 4,
  15: 5,
  16: 3,
  17: 5,
  18: 3,
  19: 3,
  20: 5,
  21: 3,
  22: 5,
  23: 5,
  24: 3,
  25: 5,
  26: 3,
  27: 5,
  28: 3,
  29: 4,
  30: 3,
  31: 3,
  32: 5,
  33: 5,
  34: 4,
  35: 5,
  36: 3,
  37: 5,
  38: 5,
  39: 3,
  40: 3,
  41: 3,
  42: 5,
  43: 4,
  44: 4,
  45: 5,
  46: 3,
  47: 5,
  48: 3,
  49: 3,
  50: 5,
  51: 5,
  52: 5,
  53: 3,
  54: 3,
  55: 5,
  56: 3,
  57: 5,
  58: 4,
  59: 4,
  60: 3,
  61: 3,
  62: 3,
  63: 5,
  64: 4,
  65: 5,
  66: 3,
  67: 3,
  68: 3,
  69: 3,
  70: 3,
  71: 3,
  72: 5,
  73: 3,
  74: 3,
  75: 3,
  76: 3,
  77: 5,
  78: 3,
  79: 3,
  80: 3,
  81: 5,
  82: 3,
  83: 3,
  84: 3,
  85: 3,
  86: 3,
  87: 5,
  88: 3,
  89: 3,
  90: 3,
  91: 3,
  92: 3,
  93: 5,
  94: 3,
  95: 5,
  96: 

## Проверка работы api

In [19]:
from src.utils.prompt import build_full_prompt
from langchain_core.prompts import ChatPromptTemplate
prompt = build_full_prompt(base_genotype, task, participant)

In [20]:
prompt_template = ChatPromptTemplate.from_messages([
        ("system", prompt["system"]),
        ("human", prompt["human"])
    ])
response = model.generate(prompt_template)

In [21]:
from src.utils.parse import parse_response
model_answers = parse_response(response.content) 

In [22]:
model_answers

{1: 2,
 2: 1,
 3: 5,
 4: 2,
 5: 5,
 6: 2,
 7: 1,
 8: 5,
 9: 4,
 10: 5,
 11: 2,
 12: 5,
 13: 2,
 14: 3,
 15: 5,
 16: 4,
 17: 4,
 18: 3,
 19: 2,
 20: 5,
 21: 3,
 22: 4,
 23: 5,
 24: 3,
 25: 5,
 26: 2,
 27: 4,
 28: 3,
 29: 3,
 30: 2,
 31: 2,
 32: 4,
 33: 5,
 34: 2,
 35: 5,
 36: 2,
 37: 1,
 38: 5,
 39: 4,
 40: 2,
 41: 2,
 42: 5,
 43: 3,
 44: 3,
 45: 5,
 46: 4,
 47: 4,
 48: 3,
 49: 2,
 50: 5,
 51: 5,
 52: 4,
 53: 3,
 54: 3,
 55: 5,
 56: 2,
 57: 4,
 58: 3,
 59: 3,
 60: 2,
 61: 2,
 62: 4,
 63: 5,
 64: 2,
 65: 5,
 66: 2,
 67: 5,
 68: 2,
 69: 4,
 70: 2,
 71: 2,
 72: 5,
 73: 3,
 74: 3,
 75: 2,
 76: 4,
 77: 4,
 78: 3,
 79: 2,
 80: 2,
 81: 5,
 82: 3,
 83: 3,
 84: 3,
 85: 2,
 86: 2,
 87: 5,
 88: 3,
 89: 3,
 90: 2,
 91: 2,
 92: 5,
 93: 5,
 94: 2,
 95: 5,
 96: 5,
 97: 5,
 98: 2,
 99: 2,
 100: 2,
 101: 5,
 102: 4,
 103: 3,
 104: 3,
 105: 2,
 106: 4,
 107: 2,
 108: 3,
 109: 2,
 110: 2,
 111: 5,
 112: 3,
 113: 3,
 114: 3,
 115: 2,
 116: 5,
 117: 5,
 118: 3,
 119: 3,
 120: 2}